# Module 43 — Object Segmentation

## 6.1 FCN — Fully Convolutional Network

**Must Know · Semantic Segmentation**

Let's understand FCN from the perspective of a Deep Learning engineer: why it was introduced, how its architecture works, how it produces pixel-level predictions, and what happens internally.

---

## 1. Why Do We Need FCN?

First, recall how a traditional CNN classification model works.

Suppose we provide an image of a dog.

![Traditional CNN image classification workflow](https://images.openai.com/static-rsc-4/9iBm1SRuj3MIsanIWtVeSKMavsw4Gyuw0Av2o698RTvyAZ4M0aid6qe2eovz3Ag00TR2vAOGOIWqcQhBKRo0TBI27R8whHUzmmPQY_cj6W5NbtQR3gsFtsenzoBUOgI-rnPQZmyEz73z_-sPlait3eNQ2Bx3l53ZL8QRBqP1_N8?purpose=inline)

*Original reference: [What Is Deep Learning? — Roboflow](https://blog.roboflow.com/deep-learning/?utm_source=chatgpt.com)*

The model processes the image and produces one prediction:

```text
Input Image
     ↓
Convolution Layers
     ↓
Pooling Layers
     ↓
Feature Extraction
     ↓
Fully Connected Layers
     ↓
Softmax
     ↓
Prediction: DOG
```

This works for image classification.

But imagine an autonomous vehicle camera.

We don't just want to know whether the image contains a car. We want to identify exactly which pixels belong to:

- Road
- Cars
- Pedestrians
- Buildings
- Trees
- Sky

This is semantic segmentation.

**Problem:** Traditional fully connected layers generally collapse spatial feature information into a fixed-size feature vector. They are not naturally designed to produce a spatial prediction for every pixel.

**Solution:** Fully Convolutional Networks (FCN).

## 2. What Is a Fully Convolutional Network?

FCN is a CNN architecture designed to perform dense, pixel-level prediction.

Its key idea is:

> Replace fully connected layers with convolutional layers so that the network can produce spatial output maps.

![Fully Convolutional Network architecture showing image input and pixel-wise segmentation output](https://www.cv-foundation.org/openaccess/content_cvpr_2015/papers/Long_Fully_Convolutional_Networks_2015_CVPR_paper.pdf)

*Source: [Fully Convolutional Networks for Semantic Segmentation — Original CVPR 2015 Paper](https://openaccess.thecvf.com/content_cvpr_2015/papers/Long_Fully_Convolutional_Networks_2015_CVPR_paper.pdf)*

**Original reference:**

- [11. Detection and Segmentation — gaussian37](https://images.openai.com/static-rsc-4/4pnT1D2CmneiyU5GNt0tkq9Sb0U0APS9bMY0THtSS0BN2vLbhapfXpsEbvQiel93c9g1cO4bxVkQsJg4eGFW6s3BCVgiZjLsJKzixHur5O7mq3lMsi7jdKDwrSJf6TYFZA5c1GQ-kjMtXVSyltIYdAQhyCOC3bb37lpqECJT2OA?purpose=inline)
- [gaussian37.github.io](https://gaussian37.github.io/vision-cs231n-detection-and-segmentation/?utm_source=chatgpt.com)

Unlike traditional classification networks, FCN preserves spatial feature-map structure throughout its prediction pipeline.

### Traditional CNN vs FCN

| Traditional CNN | FCN |
|---|---|
| Image classification | Semantic segmentation |
| Fully connected classification head | Convolutional prediction head |
| Usually fixed-size output | Spatial output map |
| Predicts image-level class | Predicts class for each pixel |

## 3. FCN Architecture — Step by Step

The architecture can be understood through five stages.

![FCN encoder and decoder architecture showing feature extraction and upsampling](https://images.openai.com/static-rsc-4/RdTkUYGhCGia7wY3ZyMfxioFVIT7n9c-WrWvbZk1Yz3rsa3uDrfUjUyU0v_k402LQAugrVDFBu1SIaLL8_Wn8vLCJpETeqerGt6G45EEmFsxfdI_QTMgT2XevLB8zcMkCUNOP2BEuAfLQgCawYQ4kvAjXnpQr1ZRCsQwkCKm8qw?purpose=inline)

*Original reference: [Coding an Encoder-Decoder Architecture for Image Segmentation — LinkedIn](https://www.linkedin.com/pulse/12-coding-encoder-decoder-architecture-image-riya-chhikara-nihje?utm_source=chatgpt.com)*

### Stage 1 — Input Image

Let's assume the input image has dimensions:

$$
H \times W \times 3
$$

For example:

```text
Input image: 256 × 256 × 3
```

Here:

- 256 = Image height
- 256 = Image width
- 3 = RGB channels

### Stage 2 — Feature Extraction

The input passes through convolution and pooling layers.

The convolution layers learn visual features:

| Layer depth | Features learned |
|---|---|
| Early layers | Edges, corners |
| Intermediate layers | Textures, patterns |
| Deep layers | Shapes, object-level features |

Pooling progressively reduces spatial resolution.

Example feature-map progression:

```text
Input:        256 × 256 × 3
                  ↓
Conv layers:  256 × 256 × 64
                  ↓
Pooling:      128 × 128 × 64
                  ↓
Conv layers:  128 × 128 × 128
                  ↓
Pooling:       64 × 64 × 128
                  ↓
Deep features: 16 × 16 × 512
```

These dimensions are illustrative, not a fixed FCN architecture.

**Important:** As the network becomes deeper, it generally captures more semantic information, but spatial detail decreases.

### Stage 3 — 1×1 Convolution

This is an important component of FCN.

Instead of flattening the feature maps and using fully connected layers, FCN applies a 1×1 convolution.

**What does it do?**

Suppose the extracted feature map is:

```python
[B, 512, 16, 16]
```

We have a segmentation problem with four classes:

```python
0 → Background
1 → Road
2 → Car
3 → Person
```

Apply a 1×1 convolution with four output channels:

```text
Input:
[B, 512, 16, 16]

       ↓
1×1 Convolution
       ↓

Output:
[B, 4, 16, 16]
```

Now, each spatial location contains four class scores.

Mathematically:

$$
Y_{b,c,h,w} =
\sum_{k=1}^{512} W_{c,k}X_{b,k,h,w}+b_c
$$

Where:

- \\(X\\) = Input feature map
- \\(W\\) = Convolution weights
- \\(b\\) = Bias
- \\(c\\) = Output class

Think of it this way: The 1×1 convolution converts the extracted feature channels into class-score channels at every spatial location.

### Stage 4 — Upsampling

We now have a class-score map of size:

```python
[B, 4, 16, 16]
```

But the original image was:

```python
[B, 3, 256, 256]
```

We need to increase the spatial resolution of the score map.

This is called **upsampling**.

![FCN upsampling process and skip connections](https://user-images.githubusercontent.com/46951365/92303325-a8966d00-efae-11ea-82b4-4db6244d5848.png)

*Source: [FCN architecture diagram — based on the original FCN paper](https://openaccess.thecvf.com/content_cvpr_2015/papers/Long_Fully_Convolutional_Networks_2015_CVPR_paper.pdf)*

One common method is transposed convolution.

It learns how to increase spatial resolution.

Other methods include:

- Bilinear interpolation
- Nearest-neighbour interpolation
- Learned upsampling through transposed convolution

After upsampling:

```text
[B, 4, 16, 16]
        ↓
    Upsampling
        ↓
[B, 4, 256, 256]
```

### Stage 5 — Pixel-wise Classification

The final output contains four class scores for every pixel.

We select the class with the highest score.

$$
\hat{y}_{h,w}=\arg\max_c S_{c,h,w}
$$

For example:

```text
Pixel (10, 20):

Background = 0.10
Road       = 0.15
Car        = 0.70
Person     = 0.05

Predicted class = Car
```

Repeating this for every pixel generates the segmentation mask.

![Semantic segmentation output showing pixel-level classification](https://images.openai.com/static-rsc-4/Nz7HGbg8gOyZA81iieOj9yDKN3g5LBvnciu8_-245788JJPDsjr1R2LzOGMXA_04ltCmCwstnrzZ8bJ4KNwcxsfBSvBTN7s3yb2gGWNUx16T2_x-JydX8RzR3blGar7S1-LuTJhfTJB3Tej10xnyoVGoEQS3k4CHJ-cSNIQh14c?purpose=inline)

*Original reference: [Semantic Segmentation: A TensorFlow Exploration of FCN — Medium](https://medium.com/%40sepideh.92sh/semantic-segmentation-a-tensorflow-exploration-of-fcn-and-transfer-learning-ecca31c624c3?utm_source=chatgpt.com)*

## 4. FCN Variants — FCN-32s, FCN-16s, FCN-8s

This is an important part of FCN architecture.

The original FCN work introduced different variants to improve spatial prediction.

The key difference is how the network combines coarse deep predictions with earlier feature maps.

![Original FCN architecture showing FCN-32s, FCN-16s and FCN-8s feature fusion](https://user-images.githubusercontent.com/46951365/92303325-a8966d00-efae-11ea-82b4-4db6244d5848.png)

*Source: [Long et al. — Fully Convolutional Networks for Semantic Segmentation, CVPR 2015](https://openaccess.thecvf.com/content_cvpr_2015/papers/Long_Fully_Convolutional_Networks_2015_CVPR_paper.pdf)*

### FCN-32s

- Uses the final coarse score map.
- Upsamples it by a factor of 32.
- Does not fuse intermediate score maps.

```text
Deep Feature Map
       ↓
Class Score Map
       ↓
  32× Upsampling
       ↓
Segmentation Mask
```

**Limitation:** Predictions can be relatively coarse, especially around object boundaries.

### FCN-16s

FCN-16s combines:

- Final deep score map.
- Intermediate score map from an earlier pooling stage.

```text
Deep Score Map
      +
Earlier Score Map
      ↓
    Fusion
      ↓
  16× Upsampling
      ↓
Segmentation Mask
```

This helps recover spatial details.

### FCN-8s

FCN-8s combines information from multiple stages.

```text
Deep Score Map
      +
Intermediate Score Map
      +
Earlier Score Map
      ↓
    Fusion
      ↓
   8× Upsampling
      ↓
Segmentation Mask
```

It generally produces finer predictions than FCN-32s and FCN-16s.

### Comparison

| Variant | Upsampling factor | Feature fusion |
|---|---:|---|
| FCN-32s | 32× | Final score map |
| FCN-16s | 16× | Final + intermediate |
| FCN-8s | 8× | Final + multiple earlier maps |

**Interview point:** Lower upsampling factors in these FCN variants are achieved by incorporating earlier, higher-resolution feature information. They are not simply different interpolation settings.

## 5. FCN Training — How Does It Learn?

FCN is trained using labelled segmentation masks.

For example:

![Original image, ground-truth mask and predicted segmentation output](https://images.openai.com/static-rsc-4/nHB81AymdNmlqTWeUaZw6gYI1Mg7896RazflYtOmIEsOTssw_x7FTY25RaUDhNwWraZsusTCj6Gk6qsQE8gqNxvmN6Ud95OPCgUYlicXRqUTqrYpNTcqSFZokraob3vWdy7IN-uYsWRvzaGEhRnH0Msvoz2fSc-PATRLjIw9LV4?purpose=inline)

*Original reference: [Faster and More Effective Scene Understanding — University of Freiburg](https://news.vm.uni-freiburg.de/en/newsarchive/faster-and-more-effective-scene-understanding?utm_source=chatgpt.com)*

We provide two things:

1. Input image.
2. Ground-truth segmentation mask.

The model predicts a class for every pixel.

We compare the predicted class scores with the ground-truth class labels using pixel-wise Cross Entropy Loss.

$$
L = -\frac{1}{HW}
\sum_{h=1}^{H}
\sum_{w=1}^{W}
\log P(y_{h,w}\mid x)
$$

Here:

- \\(H,W\\) = Image height and width.
- \\(y_{h,w}\\) = Ground-truth class for a pixel.
- \\(P\\) = Predicted probability of that class.

The optimizer updates the network weights through backpropagation.

**Important:** During training, the network uses class scores (logits) and the ground-truth mask. We generally don't apply `argmax` before `CrossEntropyLoss` because it would remove the differentiable class-score information.

## 6. Advantages and Limitations

| Advantages | Limitations |
|---|---|
| Supports pixel-level prediction | Coarse predictions in early variants |
| Fully convolutional design | Pooling loses spatial detail |
| Can process different image sizes, subject to architecture constraints | Small objects can be difficult |
| Skip fusion improves localization | Boundary precision can still be limited |

## 7. Industry Applications

![FCN-based semantic segmentation applications](https://images.openai.com/static-rsc-4/OcbT6RGgd1LG4MOeB5axAVle6bhgb2LfudfBPMJF3Q5UlJsIs783PpAwE6uiuT44NVPFNDDm0bXZjz0CwGgSLELqfzv-Kky8Mba84FsPu1QhB6OfzJ4lV8wI7agHe8NJR7O99h-9X4Dpw2qgXJ8WO-Ql3zmSCpReTS9rEaa5a9o?purpose=inline)

**Autonomous driving**

- Road, vehicle and pedestrian segmentation.

**Satellite imagery**

![Satellite image segmentation application](https://images.openai.com/static-rsc-4/3cyDgaNXHYQPr50YXxvmlUv7ta3LPdPVIYbqjXpAtHs5m5YAgMoG9sVT_IFlATTnV41fWZvDIEle7jlBRxopmeXcJsAIkg4kclkSxAVcgytTcFydVwWh6UDbflmgsNnyRO7X7fcGI1uAPkZHc4miLjreYPZxJvC1In7uhDsXKUU?purpose=inline)

- Roads, buildings and land-cover segmentation.

**Medical imaging**

![Brain MRI segmentation using deep learning](https://images.openai.com/static-rsc-4/7j-cTCNlh1bN54oOOceXPfSzDxUSPj3E_17-d7agxtTU_Yh6BYQnIONaLwcYoO4jVtslNKb5sXzT8qKSp4VpPTm_t6L_pcBtW9iGdB86McxCLFfgjyOOgBfArgIAgYCNyC-wiiyWjyvtqk0q-YUfzeaWgqAkMjNEg00oagvyjyc?purpose=inline)

- Organ and tissue segmentation.

## 8. What You Should Remember 🧠

1. FCN stands for Fully Convolutional Network.
2. It replaces fully connected classification layers with convolutional prediction layers.
3. A 1×1 convolution produces class-score maps.
4. Upsampling restores spatial resolution.
5. FCN-32s, FCN-16s and FCN-8s progressively incorporate earlier spatial features.
6. FCN is primarily used for semantic segmentation.

---